In [40]:
# Basic Libraries
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np

# PyTorch and Transformers
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel
from transformers import logging
logging.set_verbosity_error()  # Suppress warnings from the Transformers library

# Utility
from tqdm import tqdm

In [41]:

import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [42]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [43]:
class MultiTaskDataset(Dataset):
    def __init__(self, data, tokenizer, max_length):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        text = row['text']
        labels = {
            "is_fake": row['is_fake'] if row['is_fake'] != -1 else None,
            "is_toxic": row['is_toxic'] if row['is_toxic'] != -1 else None,
            "is_hate_speech": row['is_hate_speech'] if row['is_hate_speech'] != -1 else None
        }
        encoded = self.tokenizer(
            text, 
            padding="max_length", 
            truncation=True, 
            max_length=self.max_length, 
            return_tensors="pt"
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(0).to(device),
            "attention_mask": encoded["attention_mask"].squeeze(0).to(device),
            "labels": labels
        }

In [44]:
def train_model(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    task_losses = {"is_fake": 0, "is_toxic": 0, "is_hate_speech": 0}
    task_counts = {"is_fake": 0, "is_toxic": 0, "is_hate_speech": 0}

    for batch in dataloader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]

        # Forward pass
        fake_news_pred, toxicity_pred, hate_speech_pred = model(input_ids, attention_mask)

        # Initialize loss as a zero tensor on the correct device with requires_grad=True
        loss = torch.zeros(1, device=device, requires_grad=True)

        # Process Fake News task
        if labels["is_fake"] is not None:
            target_fake = torch.tensor(labels["is_fake"], device=device).float().unsqueeze(1)
            fake_news_pred = fake_news_pred.view(-1, 1)  # Ensure shape [batch_size, 1]
            if target_fake.shape == fake_news_pred.shape:
                loss_fake = criterion(fake_news_pred, target_fake)
                loss += loss_fake
                task_losses["is_fake"] += loss_fake.item()
                task_counts["is_fake"] += target_fake.size(0)

        # Process Toxicity task
        if labels["is_toxic"] is not None:
            target_toxic = torch.tensor(labels["is_toxic"], device=device).float().unsqueeze(1)
            toxicity_pred = toxicity_pred.view(-1, 1)  # Ensure shape [batch_size, 1]
            if target_toxic.shape == toxicity_pred.shape:
                loss_toxic = criterion(toxicity_pred, target_toxic)
                loss += loss_toxic
                task_losses["is_toxic"] += loss_toxic.item()
                task_counts["is_toxic"] += target_toxic.size(0)

        # Process Hate Speech task
        if labels["is_hate_speech"] is not None:
            target_hate = torch.tensor(labels["is_hate_speech"], device=device).float().unsqueeze(1)
            hate_speech_pred = hate_speech_pred.view(-1, 1)  # Ensure shape [batch_size, 1]
            if target_hate.shape == hate_speech_pred.shape:
                loss_hate = criterion(hate_speech_pred, target_hate)
                loss += loss_hate
                task_losses["is_hate_speech"] += loss_hate.item()
                task_counts["is_hate_speech"] += target_hate.size(0)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_losses = {
        task: (task_losses[task] / task_counts[task]) if task_counts[task] > 0 else 0
        for task in task_losses
    }

    return total_loss / len(dataloader), avg_losses


In [45]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
def evaluate_model(model, dataloader, criterion, device):
    model.eval()
    task_losses = {"is_fake": 0, "is_toxic": 0, "is_hate_speech": 0}
    task_counts = {"is_fake": 0, "is_toxic": 0, "is_hate_speech": 0}
    predictions = {"is_fake": [], "is_toxic": [], "is_hate_speech": []}
    ground_truths = {"is_fake": [], "is_toxic": [], "is_hate_speech": []}
    accuracies = {}

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"]
            attention_mask = batch["attention_mask"]
            labels = batch["labels"]

            fake_news_pred, toxicity_pred, hate_speech_pred = model(input_ids, attention_mask)
            batch_size = input_ids.size(0)
            
            if labels["is_fake"] is not None:
                target_fake = torch.tensor(labels["is_fake"], device=device).float().view(batch_size, 1)
                fake_news_pred = fake_news_pred.view(batch_size, 1)
                loss_fake = criterion(fake_news_pred, target_fake)
                task_losses["is_fake"] += loss_fake.item()
                task_counts["is_fake"] += 1
                pred_fake_binary = (torch.sigmoid(fake_news_pred) > 0.5).cpu().numpy()
                predictions["is_fake"].extend(pred_fake_binary)
                ground_truths["is_fake"].extend(target_fake.cpu().numpy())

            if labels["is_toxic"] is not None:
                target_toxic = torch.tensor(labels["is_toxic"], device=device).float().view(batch_size, 1)
                toxicity_pred = toxicity_pred.view(batch_size, 1)
                loss_toxic = criterion(toxicity_pred, target_toxic)
                task_losses["is_toxic"] += loss_toxic.item()
                task_counts["is_toxic"] += 1
                pred_toxic_binary = (torch.sigmoid(toxicity_pred) > 0.5).cpu().numpy()
                predictions["is_toxic"].extend(pred_toxic_binary)
                ground_truths["is_toxic"].extend(target_toxic.cpu().numpy())

            if labels["is_hate_speech"] is not None:
                target_hate = torch.tensor(labels["is_hate_speech"], device=device).float().view(batch_size, 1)
                hate_speech_pred = hate_speech_pred.view(batch_size, 1)
                loss_hate = criterion(hate_speech_pred, target_hate)
                task_losses["is_hate_speech"] += loss_hate.item()
                task_counts["is_hate_speech"] += 1
                pred_hate_binary = (torch.sigmoid(hate_speech_pred) > 0.5).cpu().numpy()
                predictions["is_hate_speech"].extend(pred_hate_binary)
                ground_truths["is_hate_speech"].extend(target_hate.cpu().numpy())
    
    avg_losses = {task: task_losses[task] / task_counts[task] for task in task_losses if task_counts[task] > 0}

    # Calculate metrics for each task
    metrics_table = []
    for task in ["is_fake", "is_toxic", "is_hate_speech"]:
        y_true = ground_truths[task]
        y_pred = predictions[task]

        accuracy = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)

        metrics_table.append({
            "Task": task,
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1-Score": f1
        })

    # Convert metrics to DataFrame
    metrics_df = pd.DataFrame(metrics_table)
    return avg_losses, metrics_df

    # # Compute accuracy for each task
    # for task in predictions.keys():
    #     if ground_truths[task]:
    #         accuracies[task] = accuracy_score(ground_truths[task], predictions[task])
    
    # return avg_losses, predictions, ground_truths, accuracies

In [46]:
# MultiTaskModel Definition
class MultiTaskModel(nn.Module):
    def __init__(self, model_name, task_outputs):
        """
        Initializes the MultiTaskModel.
        
        Args:
            model_name (str): The name of the pretrained model to use (e.g., "bert-base-uncased").
            task_outputs (dict): A dictionary specifying the number of outputs for each task.
                                 Example: {"is_fake": 1, "is_toxic": 1, "is_hate_speech": 1}.
        """
        super(MultiTaskModel, self).__init__()
        # Load the pretrained transformer model
        self.shared_model = AutoModel.from_pretrained(model_name)
        self.task_heads = nn.ModuleDict({
            task: nn.Linear(self.shared_model.config.hidden_size, output_size)
            for task, output_size in task_outputs.items()
        })

    def forward(self, input_ids, attention_mask):
        """
        Forward pass of the model.
        
        Args:
            input_ids (torch.Tensor): Input token IDs.
            attention_mask (torch.Tensor): Attention masks for the input.

        Returns:
            Tuple of outputs for each task.
        """
        # Extract the shared representation from the transformer model
        shared_output = self.shared_model(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = shared_output.pooler_output  # Use the [CLS] token representation
        
        # Pass the shared representation to each task head
        task_outputs = {task: head(pooled_output) for task, head in self.task_heads.items()}
        
        return task_outputs["is_fake"], task_outputs["is_toxic"], task_outputs["is_hate_speech"]


In [47]:

df_combined = pd.read_csv("/home/s2shsinh/TWON_Metrics/dataset/merged_dataset_4jan.csv")
train_data, val_data = train_test_split(df_combined, test_size=0.2, random_state=42)

   

In [48]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    # Filter out None samples
    batch = [b for b in batch if b is not None]
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "labels": {key: [b["labels"][key] for b in batch if b["labels"][key] is not None] for key in batch[0]["labels"]}
    }

In [49]:
tokenizer = AutoTokenizer.from_pretrained("Twitter/twhin-bert-base")
train_dataset = MultiTaskDataset(train_data, tokenizer, max_length=128)
val_dataset = MultiTaskDataset(val_data, tokenizer, max_length=128)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=16, collate_fn=collate_fn)

# Initialize model and optimizer
model = MultiTaskModel("Twitter/twhin-bert-base", {"is_fake": 1, "is_toxic": 1, "is_hate_speech": 1}).to(device)
optimizer = AdamW(model.parameters(), lr=2e-5)
criterion = nn.BCEWithLogitsLoss()



In [50]:
model

MultiTaskModel(
  (shared_model): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
              (distance_embedding): Embedding(1023, 64)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
 

In [51]:
train_dataset

In [52]:
# Training loop
for epoch in range(3):
    train_loss, train_task_losses = train_model(model, train_loader, optimizer, criterion, device)
    val_task_losses, metrics_df = evaluate_model(model, val_loader, criterion, device)
    
    print(f"Epoch {epoch+1}, Train Loss: {train_loss}, Task Losses: {train_task_losses}")
    print(f"Validation Task Losses: {val_task_losses}")
    print("Validation Metrics:")
    print(metrics_df)

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn